In [1]:
import pandas as pd
import numpy as np
import logging
from sklearn.model_selection import train_test_split

# 1. Initialize our module-level logger (consistent with your Part 2 pipeline)
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Load the feature-engineered dataset output from Part 2
try:
    df_ml = pd.read_csv("engineered_traffic_data.csv")
    logger.info(f"Successfully loaded engineered dataset for ML. Initial shape: {df_ml.shape}")
except FileNotFoundError:
    # Helper fallback in case your output file is named slightly differently
    df_ml = df.copy() if 'df' in locals() else pd.read_csv("cleaned_traffic_data.csv")
    logger.warning("Engineered dataset file not found. Falling back to active memory/cleaned frame.")

print("--- STEP 1: CONSTRUCTING DATA-DRIVEN PROXY LABELS ---")

# 2. Calculate data-driven quartile thresholds for traffic volume
q1, q2, q3 = df_ml["traffic_volume"].quantile([0.25, 0.5, 0.75]).values
logger.info(f"Traffic Volume Quartiles calculated - Q1: {q1:.1f}, Q2: {q2:.1f}, Q3: {q3:.1f}")
print(f"[INFO] Traffic Quartile Thresholds -> Q1: {q1:.0f}, Q2: {q2:.0f}, Q3: {q3:.0f} vehicles")

# 3. Define the quartile bucketing function requested by the assignment
def bucket_congestion(v):
    if v <= q1:
        return "Low"
    elif v <= q2:
        return "Medium"
    elif v <= q3:
        return "High"
    return "Severe"

# Apply the quartile splitting logic to create our congestion category
df_ml["congestion_category"] = df_ml["traffic_volume"].apply(bucket_congestion)

# 4. Define our severe weather criteria and construct the binary accident-risk proxy
# (Adjust these weather names if your Part 2 cleanup text-formatting varies)
SEVERE_WEATHER = ["Rain", "Snow", "Thunderstorm", "Squall"]

# A condition is severe/low-visibility if it matches severe terms OR clouds/rain indicator flags
risky_weather = (
    df_ml["weather_main"].isin(SEVERE_WEATHER) | 
    (df_ml.get("is_low_visibility", 0) == 1) | 
    (df_ml.get("is_raining", 0) == 1)
)

# High congestion means the quartile bucket falls into 'High' or 'Severe'
high_congestion = df_ml["congestion_category"].isin(["High", "Severe"])

# Combine the matrices: High Risk (1) = High Congestion AND Risky Weather
df_ml["high_risk"] = (high_congestion & risky_weather).astype(int)

# 5. Audit and print out the distribution profile of our new labels
risk_counts = df_ml["high_risk"].value_counts()
risk_percentage = df_ml["high_risk"].value_counts(normalize=True) * 100

logger.info(f"Proxy accident-risk label generated. Balance: Low Risk={risk_counts.get(0, 0)}, High Risk={risk_counts.get(1, 0)}")

print(f"\n[AUDIT] Proxy Accident-Risk Label Distribution:")
print(f" -> Normal/Low Risk (0): {risk_counts.get(0, 0)} records ({risk_percentage.get(0, 0):.2f}%)")
print(f" -> High Accident Risk (1): {risk_counts.get(1, 0)} records ({risk_percentage.get(1, 0):.2f}%)")


--- STEP 1: CONSTRUCTING DATA-DRIVEN PROXY LABELS ---
[INFO] Traffic Quartile Thresholds -> Q1: 1192, Q2: 3379, Q3: 4933 vehicles

[AUDIT] Proxy Accident-Risk Label Distribution:
 -> Normal/Low Risk (0): 42947 records (89.13%)
 -> High Accident Risk (1): 5240 records (10.87%)


In [12]:
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)

# 1. Ensure holiday string is cleaned into a binary flag (0 or 1)
df_ml['holiday'] = df_ml['holiday'].apply(lambda x: 0 if str(x).strip() == 'None' else 1)

# 2. Hardcode a whitelist of ONLY the numeric feature columns we want
feature_cols = [
    'holiday', 'is_weekend',
    'hour_sin', 'hour_cos', 'day_sin', 'day_cos',  # Cyclical time encodings
    'temp_scaled', 'clouds_scaled',                 # Min-Max scaled features
    'is_freezing', 'is_raining'                     # Derived weather indicator flags
]

# 3. Safely include only the true one-hot encoded numeric dummy headers (0 or 1 columns)
# This ignores 'weather_main' and 'weather_description' because they do not start with 'weather_' + a dynamic split category name
weather_dummy_cols = [col for col in df_ml.columns if col.startswith('weather_') and col not in ['weather_main', 'weather_description']]
feature_cols.extend(weather_dummy_cols)

# Separate absolute numbers features (X) and classification target (y)
X_class = df_ml[feature_cols].copy()
y_class = df_ml['high_risk'].copy()

# Print out data types to guarantee it's 100% text-free
print("--- VERIFIED FEATURE DATA TYPES AUDIT ---")
print(X_class.dtypes)
print(f"\nFinal Safe Feature Set Shape: {X_class.shape}")


--- VERIFIED FEATURE DATA TYPES AUDIT ---
holiday                   int64
is_weekend                int64
hour_sin                float64
hour_cos                float64
day_sin                 float64
day_cos                 float64
temp_scaled             float64
clouds_scaled           float64
is_freezing               int64
is_raining                int64
weather_Clear             int64
weather_Clouds            int64
weather_Drizzle           int64
weather_Fog               int64
weather_Haze              int64
weather_Mist              int64
weather_Rain              int64
weather_Smoke             int64
weather_Snow              int64
weather_Squall            int64
weather_Thunderstorm      int64
dtype: object

Final Safe Feature Set Shape: (48187, 21)


In [13]:
from sklearn.model_selection import train_test_split

# Re-split with the clean, numeric-only arrays
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.2, random_state=42, stratify=y_class
)

print("--- DATA SPLIT VERIFICATION ---")
print(f"Training Features Shape: {X_train_c.shape}")
print(f"Testing Features Shape : {X_test_c.shape}")


--- DATA SPLIT VERIFICATION ---
Training Features Shape: (38549, 21)
Testing Features Shape : (9638, 21)


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# 1. Train Baseline Logistic Regression
baseline_logreg = LogisticRegression(max_iter=1000, random_state=42)
baseline_logreg.fit(X_train_c, y_train_c)

# 2. Train Ensemble Random Forest Classifier
ensemble_rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
ensemble_rf.fit(X_train_c, y_train_c)

# 3. Generate Predictions and Probabilities
y_pred_logreg = baseline_logreg.predict(X_test_c)
y_prob_logreg = baseline_logreg.predict_proba(X_test_c)[:, 1]

y_pred_rf = ensemble_rf.predict(X_test_c)
y_prob_rf = ensemble_rf.predict_proba(X_test_c)[:, 1]

# 4. Print Comparison Metrics Panel
print("==================================================")
print("CLASSIFICATION MODEL 1: LOGISTIC REGRESSION (BASELINE)")
print("==================================================")
print(f"Accuracy  : {accuracy_score(y_test_c, y_pred_logreg):.4f}")
print(f"Precision : {precision_score(y_test_c, y_pred_logreg):.4f}")
print(f"Recall    : {recall_score(y_test_c, y_pred_logreg):.4f}")
print(f"F1-Score  : {f1_score(y_test_c, y_pred_logreg):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test_c, y_prob_logreg):.4f}")

print("\n==================================================")
print("CLASSIFICATION MODEL 2: RANDOM FOREST (ENSEMBLE)")
print("==================================================")
print(f"Accuracy  : {accuracy_score(y_test_c, y_pred_rf):.4f}")
print(f"Precision : {precision_score(y_test_c, y_pred_rf):.4f}")
print(f"Recall    : {recall_score(y_test_c, y_pred_rf):.4f}")
print(f"F1-Score  : {f1_score(y_test_c, y_pred_rf):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test_c, y_prob_rf):.4f}")


CLASSIFICATION MODEL 1: LOGISTIC REGRESSION (BASELINE)
Accuracy  : 0.9741
Precision : 0.8990
Recall    : 0.8578
F1-Score  : 0.8779
ROC AUC   : 0.9910

CLASSIFICATION MODEL 2: RANDOM FOREST (ENSEMBLE)
Accuracy  : 0.9901
Precision : 0.9517
Recall    : 0.9580
F1-Score  : 0.9548
ROC AUC   : 0.9980


In [15]:
import logging
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

logger = logging.getLogger(__name__)

print("--- STEP 4: TRAINING REGRESSION MODELS FOR TRAFFIC VOLUME ---")

# 1. Isolate continuous traffic volume as our numeric target (y)
# We reuse the exact same clean, numeric features (X_class) from your previous steps
X_reg = X_class.copy()
y_reg = df_ml['traffic_volume'].copy()

# 2. Train-Test Split (80% Training, 20% Testing)
X_train_r, X_test_c_r, y_train_r, y_test_r = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)
logger.info(f"Regression data split complete. Train shape: {X_train_r.shape}, Test shape: {X_test_c_r.shape}")

# 3. Algorithm 1: Train Baseline Linear Regression
baseline_reg = LinearRegression()
baseline_reg.fit(X_train_r, y_train_r)
logger.info("Baseline Linear Regression model training complete.")

# 4. Algorithm 2: Train Advanced Tree-Based Ensemble (Random Forest Regressor)
# n_jobs=-1 speeds up execution by utilizing all available CPU cores
ensemble_reg = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
ensemble_reg.fit(X_train_r, y_train_r)
logger.info("Ensemble Random Forest Regressor model training complete.")

# 5. Generate continuous predictions on the test partition
y_pred_lin = baseline_reg.predict(X_test_c_r)
y_pred_rf_reg = ensemble_reg.predict(X_test_c_r)

# 6. Print out evaluation summaries for comparison
print("\n==================================================")
print("REGRESSION MODEL 1: LINEAR REGRESSION (BASELINE)")
print("==================================================")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_r, y_pred_lin):.2f} vehicles")
print(f"R-squared (R2 Score)     : {r2_score(y_test_r, y_pred_lin):.4f}")

print("\n==================================================")
print("REGRESSION MODEL 2: RANDOM FOREST REGRESSOR (ENSEMBLE)")
print("==================================================")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_r, y_pred_rf_reg):.2f} vehicles")
print(f"R-squared (R2 Score)     : {r2_score(y_test_r, y_pred_rf_reg):.4f}")


--- STEP 4: TRAINING REGRESSION MODELS FOR TRAFFIC VOLUME ---

REGRESSION MODEL 1: LINEAR REGRESSION (BASELINE)
Mean Absolute Error (MAE): 834.28 vehicles
R-squared (R2 Score)     : 0.7093

REGRESSION MODEL 2: RANDOM FOREST REGRESSOR (ENSEMBLE)
Mean Absolute Error (MAE): 274.11 vehicles
R-squared (R2 Score)     : 0.9433


In [16]:
import logging
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

logger = logging.getLogger(__name__)

print("--- TASK 2.1: K-MEANS CLUSTERING ANALYSIS ---")

# 1. Select relevant variables to cluster traffic conditions
cluster_features = ['hour', 'temp_scaled', 'clouds_scaled', 'traffic_volume']
X_cluster = df_ml[cluster_features].copy()

# 2. Standardize features to give equal weight to all columns
scaler = StandardScaler()
X_cluster_scaled = scaler.fit_transform(X_cluster)

# 3. Fit K-Means with an optimal cluster count of K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df_ml['cluster_label'] = kmeans.fit_predict(X_cluster_scaled)
logger.info("K-Means clustering trained and labels assigned.")

# 4. Generate profile metrics to interpret what each cluster represents
cluster_profiles = df_ml.groupby('cluster_label')[cluster_features].mean()
cluster_sizes = df_ml['cluster_label'].value_counts()

print("\n==================================================")
print("TRAFFIC CLUSTER PROFILES (MEAN CENTROIDS)")
print("==================================================")
for cluster_id in sorted(df_ml['cluster_label'].unique()):
    profile = cluster_profiles.loc[cluster_id]
    size = cluster_sizes.loc[cluster_id]
    print(f"\nCluster #{cluster_id} (Size: {size} records):")
    print(f" -> Time of Day (Hour) : {profile['hour']:.1f}:00")
    print(f" -> Traffic Volume     : {profile['traffic_volume']:.0f} vehicles")
    print(f" -> Cloud Cover (Scaled): {profile['clouds_scaled']:.2f}")
    
    # Provide semantic descriptions based on centroids
    if profile['traffic_volume'] > 4500:
        print(" 🎯 Profile: Daytime Daytime Peak / High-Congestion Rush Hour")
    elif profile['hour'] < 7 or profile['hour'] > 20:
        print(" 🎯 Profile: Late-Night / Early-Morning Low Volume Cruising")
    else:
        print(" 🎯 Profile: Mid-Day Steady / Moderate Traffic Conditions")


--- TASK 2.1: K-MEANS CLUSTERING ANALYSIS ---

TRAFFIC CLUSTER PROFILES (MEAN CENTROIDS)

Cluster #0 (Size: 13501 records):
 -> Time of Day (Hour) : 3.0:00
 -> Traffic Volume     : 828 vehicles
 -> Cloud Cover (Scaled): 0.44
 🎯 Profile: Late-Night / Early-Morning Low Volume Cruising

Cluster #1 (Size: 19307 records):
 -> Time of Day (Hour) : 14.5:00
 -> Traffic Volume     : 4200 vehicles
 -> Cloud Cover (Scaled): 0.83
 🎯 Profile: Mid-Day Steady / Moderate Traffic Conditions

Cluster #2 (Size: 15379 records):
 -> Time of Day (Hour) : 14.8:00
 -> Traffic Volume     : 4213 vehicles
 -> Cloud Cover (Scaled): 0.11
 🎯 Profile: Mid-Day Steady / Moderate Traffic Conditions


ModuleNotFoundError: No module named 'mlxtend'

In [21]:
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)

print("--- TASK 2.2: NATIVE ASSOCIATION PATTERN ANALYSIS (PANDAS PROXY) ---")

# 1. Discretize variables into conditional transaction buckets matching the assignment brief
df_rules = pd.DataFrame()

# Discretize hour into distinct day blocks
def get_time_block(h):
    if 6 <= h < 10: return "MorningRush"
    elif 10 <= h < 15: return "MidDay"
    elif 15 <= h < 19: return "EveningRush"
    return "Night/OffPeak"

df_rules['time_block'] = df_ml['hour'].apply(get_time_block)
df_rules['day_type'] = df_ml['is_weekend'].apply(lambda x: "Weekend" if x == 1 else "Weekday")
df_rules['weather'] = df_ml['weather_main']
df_rules['congestion'] = df_ml['congestion_category']

# Calculate the baseline probability of Severe Congestion across the entire dataset, P(B)
p_severe_baseline = (df_rules['congestion'] == 'Severe').mean()
total_records = len(df_rules)

# 2. Group the data by our overlapping structural conditions to find the cross-over frequencies
combinations = df_rules.groupby(['time_block', 'day_type', 'weather'])

rules_list = []

# 3. Mathematically compute Support, Confidence, and Lift for every overlapping group
for group_keys, group_data in combinations:
    time_b, day_t, weath = group_keys
    group_size = len(group_data)
    
    # Ignore tiny sample groups to avoid statistical noise
    if group_size < 50:
        continue
        
    # Calculate conditional parameters
    severe_in_group = (group_data['congestion'] == 'Severe').sum()
    
    support = severe_in_group / total_records     # P(A AND B)
    confidence = severe_in_group / group_size     # P(B | A)
    
    # Lift = Confidence / Baseline Probability of Severe Congestion
    lift = confidence / p_severe_baseline if p_severe_baseline > 0 else 0
    
    # Only save valid rules showing a positive upward relationship (Lift > 1)
    if lift > 1.0 and severe_in_group > 0:
        rules_list.append({
            'antecedents': f"If Time is [{time_b}] AND Day Type is [{day_t}] AND Weather is [{weath}]",
            'support': support,
            'confidence': confidence,
            'lift': lift
        })

# Convert to DataFrame to sort and display cleanly
df_native_rules = pd.DataFrame(rules_list)

print("\n==================================================")
print("MINED ASSOCIATION CONFIGURATIONS PREDICTING SEVERE CONGESTION")
print("==================================================")
if df_native_rules.empty:
    print("No meaningful high-lift configurations found above thresholds.")
else:
    # Sort by highest Lift as requested by the brief
    top_native_rules = df_native_rules.sort_values(by='lift', ascending=False).head(4)
    for idx, row in top_native_rules.reset_index(drop=True).iterrows():
        print(f"\nRule #{idx + 1}: {row['antecedents']} -> Then [Congestion: Severe]")
        print(f" -> Support   : {row['support']:.4f} (Factual probability of total occurrences)")
        print(f" -> Confidence: {row['confidence']:.4f} (Probability of severe bottleneck when condition is met)")
        print(f" -> Lift      : {row['lift']:.2f}x (Likelihood multiplier over random highway baseline)")



--- TASK 2.2: NATIVE ASSOCIATION PATTERN ANALYSIS (PANDAS PROXY) ---

MINED ASSOCIATION CONFIGURATIONS PREDICTING SEVERE CONGESTION

Rule #1: If Time is [MorningRush] AND Day Type is [Weekday] AND Weather is [Thunderstorm] -> Then [Congestion: Severe]
 -> Support   : 0.0022 (Factual probability of total occurrences)
 -> Confidence: 0.8974 (Probability of severe bottleneck when condition is met)
 -> Lift      : 3.59x (Likelihood multiplier over random highway baseline)

Rule #2: If Time is [MorningRush] AND Day Type is [Weekday] AND Weather is [Drizzle] -> Then [Congestion: Severe]
 -> Support   : 0.0047 (Factual probability of total occurrences)
 -> Confidence: 0.8964 (Probability of severe bottleneck when condition is met)
 -> Lift      : 3.59x (Likelihood multiplier over random highway baseline)

Rule #3: If Time is [MorningRush] AND Day Type is [Weekday] AND Weather is [Fog] -> Then [Congestion: Severe]
 -> Support   : 0.0038 (Factual probability of total occurrences)
 -> Confidence

In [23]:
import numpy as np
import pandas as pd
import logging
from sklearn.preprocessing import MinMaxScaler

logger = logging.getLogger(__name__)
print("--- STEP 1: RESHAPING DATA FOR SEQUENTIAL LSTM ---")

# 1. Isolate the target 'traffic_volume' and our established numerical feature matrix
# We scale the target variable between 0 and 1 explicitly to help the Neural Network converge faster
target_scaler = MinMaxScaler()
df_ml['traffic_scaled'] = target_scaler.fit_transform(df_ml[['traffic_volume']])

# Combine the scaled target with our numerical features from Task 1
lstm_features = ['traffic_scaled', 'holiday', 'is_weekend', 'hour_sin', 'hour_cos', 'temp_scaled', 'clouds_scaled']
data_matrix = df_ml[lstm_features].values

# 2. Build rolling sequence windows (Lookback of 6 hours to predict the next hour)
def create_sequences(data, lookback=6):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i : i + lookback, :]) # Features across past lookback window
        y.append(data[i + lookback, 0])      # Target: traffic_scaled at the next hour
    return np.array(X), np.array(y)

LOOKBACK_WINDOW = 6
X_seq, y_seq = create_sequences(data_matrix, lookback=LOOKBACK_WINDOW)

# 3. Train-Test Split sequentially (preserving time order for sequential evaluation)
split_idx = int(len(X_seq) * 0.8)
X_train_seq, X_test_seq = X_seq[:split_idx], X_seq[split_idx:]
y_train_seq, y_test_seq = y_seq[:split_idx], y_seq[split_idx:]

print(f"[INFO] 3D Training Sequences Shape: {X_train_seq.shape} -> (Samples, Lookback Hours, Features)")
print(f"[INFO] 3D Testing Sequences Shape : {X_test_seq.shape}")


--- STEP 1: RESHAPING DATA FOR SEQUENTIAL LSTM ---
[INFO] 3D Training Sequences Shape: (38544, 6, 7) -> (Samples, Lookback Hours, Features)
[INFO] 3D Testing Sequences Shape : (9637, 6, 7)


In [24]:
import logging
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import mean_absolute_error, r2_score

logger = logging.getLogger(__name__)
print("--- STEP 2: TRAINING DEEP LEARNING MODEL (LSTM) ---")

# 1. Define the Sequential LSTM Architecture
model_lstm = Sequential([
    # First LSTM Layer: 32 hidden units, returns sequences for the next layer
    LSTM(32, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]), return_sequences=True),
    Dropout(0.2), # Dropout layer prevents the neural network from overfitting
    
    # Second LSTM Layer: 16 hidden units, returns only the final summary vector
    LSTM(16, return_sequences=False),
    Dropout(0.2),
    
    # Dense Output Layer: Predicts the single continuous scaled traffic volume number
    Dense(1)
])

# 2. Compile the Deep Learning Model using Adam optimizer and Mean Squared Error loss
model_lstm.compile(optimizer='adam', loss='mean_squared_error')
logger.info("LSTM Model architecture compiled successfully.")

# 3. Train the model (Using 5 epochs with a batch size of 64 for efficient local execution)
print("\n[INFO] Starting LSTM Model Training...")
history = model_lstm.fit(
    X_train_seq, y_train_seq,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)
logger.info("LSTM Model training complete over 5 epochs.")

# 4. Generate predictions on the test partition
y_pred_lstm_scaled = model_lstm.predict(X_test_seq)

# 5. Inverse-scale predictions back to actual continuous vehicle counts
y_pred_lstm = target_scaler.inverse_transform(y_pred_lstm_scaled).flatten()
# Re-scale actual values for the same time window to compute real-world error metrics
y_test_lstm_actual = target_scaler.inverse_transform(y_test_seq.reshape(-1, 1)).flatten()

# 6. Print out deep learning evaluation metrics
print("\n==================================================")
print("DEEP LEARNING TIME-SERIES MODEL: INVERSE-SCALED LSTM")
print("==================================================")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test_lstm_actual, y_pred_lstm):.2f} vehicles")
print(f"R-squared (R2 Score)     : {r2_score(y_test_lstm_actual, y_pred_lstm):.4f}")


--- STEP 2: TRAINING DEEP LEARNING MODEL (LSTM) ---


C:\Users\chewd\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



[INFO] Starting LSTM Model Training...
Epoch 1/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0320 - val_loss: 0.0131
Epoch 2/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 0.0164 - val_loss: 0.0080
Epoch 3/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - loss: 0.0118 - val_loss: 0.0060
Epoch 4/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 4s 7ms/step - loss: 0.0098 - val_loss: 0.0046
Epoch 5/5
543/543 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.0088 - val_loss: 0.0043
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step

DEEP LEARNING TIME-SERIES MODEL: INVERSE-SCALED LSTM
Mean Absolute Error (MAE): 325.18 vehicles
R-squared (R2 Score)     : 0.9484


In [26]:
%pip install shap



   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   -------------------- ------------------- 1/2 [shap]
   ---------------------------------------- 2/2 [shap]

Note: you may need to restart the kernel to use updated packages.


In [28]:
%pip install --upgrade scikit-learn scipy


   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   --------------- ------------------------ 14.2/36.6 MB 74.7 MB/s eta 0:00:01
   ---------------------------- ----------- 26.0/36.6 MB 63.7 MB/s eta 0:00:01
   ---------------------------------------  36.4/36.6 MB 66.5 MB/s eta 0:00:01
   ---------------------------------------- 36.6/36.6 MB 54.3 MB/s  0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
Note: you may need to restart the kernel to use updated packages.


In [31]:
import pandas as pd
import numpy as np
import logging

logger = logging.getLogger(__name__)

print("--- TASK 2.1: TEXT-BASED EXPLAINABILITY (PANDAS PROXY) ---")

# 1. Directly extract numerical importance weights from your Random Forest model
importances = ensemble_reg.feature_importances_

# 2. Map weights to their actual feature text labels in a Pandas Series
feat_importance_series = pd.Series(importances, index=X_reg.columns)

# 3. Sort features by mathematical weight in descending order
sorted_importances = feat_importance_series.sort_values(ascending=False)

print("\n==================================================")
print("RANDOS FOREST RELATIVE FEATURE IMPORTANCE RANKINGS")
print("==================================================")
# Print the relative mathematical weight of every single variable
for rank, (feature_name, weight) in enumerate(sorted_importances.items()):
    print(f"Rank #{rank + 1:02d}: {feature_name:<20} -> Weight Impact: {weight:.4f}")

# Extract top operational driver for logging purposes
top_driver = sorted_importances.index[0]
logger.info(f"Feature importance matrix generated. Primary driver identified: {top_driver}")


--- TASK 2.1: TEXT-BASED EXPLAINABILITY (PANDAS PROXY) ---

RANDOS FOREST RELATIVE FEATURE IMPORTANCE RANKINGS
Rank #01: hour_cos             -> Weight Impact: 0.6508
Rank #02: hour_sin             -> Weight Impact: 0.1751
Rank #03: day_sin              -> Weight Impact: 0.0608
Rank #04: temp_scaled          -> Weight Impact: 0.0452
Rank #05: is_weekend           -> Weight Impact: 0.0427
Rank #06: day_cos              -> Weight Impact: 0.0081
Rank #07: clouds_scaled        -> Weight Impact: 0.0080
Rank #08: weather_Clouds       -> Weight Impact: 0.0020
Rank #09: is_raining           -> Weight Impact: 0.0013
Rank #10: weather_Mist         -> Weight Impact: 0.0011
Rank #11: weather_Snow         -> Weight Impact: 0.0010
Rank #12: weather_Fog          -> Weight Impact: 0.0009
Rank #13: weather_Clear        -> Weight Impact: 0.0008
Rank #14: weather_Haze         -> Weight Impact: 0.0007
Rank #15: weather_Rain         -> Weight Impact: 0.0006
Rank #16: weather_Thunderstorm -> Weight Impact: 

In [33]:
import os
import numpy as np
import pandas as pd
import logging

logger = logging.getLogger(__name__)
print("--- TASK 4: AI MODEL OPTIMIZATION VIA WEIGHT QUANTIZATION ---")

# FIX: Create the local data directory if it does not already exist
os.makedirs('data', exist_ok=True)
logger.info("Verified local data directory path exists.")

# 1. Isolate the mathematical weights (coefficients) from your trained Baseline Regression model
original_weights = baseline_reg.coef_
logger.info(f"Extracting baseline regression model float weights. Shape: {original_weights.shape}")

# Save the baseline weights as a standard high-precision float64 numpy file
np.save("data/baseline_float64_weights.npy", original_weights)

# 2. Perform Quantization: Scale and downcast the float64 weights into compact int8 arrays
# We compute a dynamic scale factor to map the floats into the standard int8 range (-128 to 127)
max_weight_val = np.max(np.abs(original_weights))
scale_factor = 127.0 / max_weight_val if max_weight_val > 0 else 1.0

# Scale and convert to 8-bit integers
quantized_weights_int8 = np.round(original_weights * scale_factor).astype(np.int8)

# Save the optimized, compressed int8 array to disk
np.save("data/quantized_int8_weights.npy", quantized_weights_int8)
logger.info("Successfully quantized and exported model weights as int8 array.")

# 3. Calculate and display the exact storage footprint compression matrix
float_size_bytes = os.path.getsize("data/baseline_float64_weights.npy")
int8_size_bytes = os.path.getsize("data/quantized_int8_weights.npy")
compression_ratio = float_size_bytes / int8_size_bytes if int8_size_bytes > 0 else 1

print("\n==================================================")
print("EDGE AI OPTIMIZATION: MODEL QUANTIZATION REPORT")
print("==================================================")
print(f"Original Float64 Weights File Size : {float_size_bytes} Bytes")
print(f"Quantized Int8 Weights File Size    : {int8_size_bytes} Bytes")
print(f"Achieved Model Compression Ratio    : {compression_ratio:.2f}x smaller")
print("🎯 Status: Compressed assets ready for deployment to local smart traffic cameras.")


--- TASK 4: AI MODEL OPTIMIZATION VIA WEIGHT QUANTIZATION ---

EDGE AI OPTIMIZATION: MODEL QUANTIZATION REPORT
Original Float64 Weights File Size : 296 Bytes
Quantized Int8 Weights File Size    : 149 Bytes
Achieved Model Compression Ratio    : 1.99x smaller
🎯 Status: Compressed assets ready for deployment to local smart traffic cameras.


In [34]:
import pandas as pd
import numpy as np
import logging

# 1. Initialize our module-level logger matching strict guidelines
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

print("--- TASK 5: TRAFFIC TIMING RECOMMENDATION SYSTEM INITIALIZATION ---")

def generate_travel_recommendation(day_type, weather_condition):
    """
    Transforms historical data arrays into optimal travel timing insights.
    day_type: 'Weekday' or 'Weekend'
    weather_condition: String matching main weather types (e.g., 'Clear', 'Rain', 'Snow', 'Clouds')
    """
    logger.info(f"Travel recommendation engine invoked. Args: Day_Type={day_type}, Weather={weather_condition}")
    
    # Map the user input string to our numeric baseline features
    is_weekend_val = 1 if day_type == "Weekend" else 0
    
    # Filter our processed DataFrame down to matches for this specific scenario
    filtered_df = df_ml[
        (df_ml['is_weekend'] == is_weekend_val) & 
        (df_ml['weather_main'].str.lower() == weather_condition.lower())
    ]
    
    # Fallback safety handler if the specific weather split contains no records
    if filtered_df.empty:
        logger.warning(f"No exact historical matches found for Day:{day_type} and Weather:{weather_condition}. Falling back to day-only filter.")
        filtered_df = df_ml[df_ml['is_weekend'] == is_weekend_val]
        
    # Group by hour and calculate the mean baseline traffic volume for each hour slot
    hourly_averages = filtered_df.groupby('hour')['traffic_volume'].mean()
    
    # Identify the absolute lowest-traffic hour window
    optimal_hour = hourly_averages.idxmin()
    min_volume = hourly_averages.min()
    
    # Format the hour into readable AM/PM strings
    start_time = f"{int(optimal_hour):02d}:00"
    end_time = f"{int(optimal_hour) + 1:02d}:00"
    
    # Construct the final plain-language response string requested by the prompt
    recommendation_text = (
        f"For a {day_type.lower()} journey during [{weather_condition}] conditions, "
        f"consider travelling between {start_time} and {end_time}, when historical traffic "
        f"volumes are typically lower (averaging {min_volume:.0f} vehicles)."
    )
    
    return recommendation_text

# 2. Audit and execute sample test scenarios to verify output formats
logger.info("Executing recommendation engine verification scenarios.")

# Test Scenario A: Standard Weekday commutes during Rain
recommendation_a = generate_travel_recommendation(day_type="Weekday", weather_condition="Rain")
print("\n[SYSTEM RUN] Scenario A Output:")
print(recommendation_a)

# Test Scenario B: Weekend cruise during Clear weather
recommendation_b = generate_travel_recommendation(day_type="Weekend", weather_condition="Clear")
print("\n[SYSTEM RUN] Scenario B Output:")
print(recommendation_b)


--- TASK 5: TRAFFIC TIMING RECOMMENDATION SYSTEM INITIALIZATION ---

[SYSTEM RUN] Scenario A Output:
For a weekday journey during [Rain] conditions, consider travelling between 02:00 and 03:00, when historical traffic volumes are typically lower (averaging 299 vehicles).

[SYSTEM RUN] Scenario B Output:
For a weekend journey during [Clear] conditions, consider travelling between 04:00 and 05:00, when historical traffic volumes are typically lower (averaging 384 vehicles).


In [35]:
import pandas as pd
import numpy as np
import logging
import json

# 1. Initialize our module-level logger matching guidelines
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

print("--- TASK 6: MLOPS DEPLOYMENT AND MONITORING SIMULATION ---")

# ==========================================
# 6.1 & 6.2: MOCK EXPERIMENT TRACKING MATRIX
# ==========================================
mock_mlflow_registry = {
    "experiment_name": "I94_Traffic_Intelligence",
    "runs": [
        {
            "run_id": "run_v1_baseline",
            "model_type": "Logistic Regression / Linear Baseline",
            "metrics": {"classification_f1": 0.8779, "regression_mae": 834.28}
        },
        {
            "run_id": "run_v2_ensemble",
            "model_type": "Random Forest Ensemble (Production Candidate)",
            "metrics": {"classification_f1": 0.9548, "regression_mae": 274.11}
        }
    ]
}
logger.info(f"MLflow experiment tracking log initialized. Active experiments logged: {len(mock_mlflow_registry['runs'])}")

# ==========================================
# 6.3: API ENDPOINT SERVICE MOCK-UP
# ==========================================
class TrafficModelAPIMock:
    def __init__(self, trained_model, feature_columns):
        self.model = trained_model
        self.feature_columns = feature_columns
        logger.info("Production API endpoint mock successfully initialized serving 'Random Forest Ensemble'.")

    def predict(self, json_payload):
        """Simulates a FastAPI/Flask endpoint accepting JSON input and returning JSON prediction"""
        try:
            # Parse the incoming operational string request data
            input_data = json.loads(json_payload)
            logger.info(f"API Request received at /predict endpoint. Payload: {input_data}")
            
            # Construct a single row matching our model array dimensions
            df_input = pd.DataFrame([input_data])
            
            # Ensure any missing whitelisted evaluation flags are dynamically backfilled with zeros
            for col in self.feature_columns:
                if col not in df_input.columns:
                    df_input[col] = 0
            
            # Restrict alignment array matrix columns to match feature matrix shape exactly
            X_api = df_input[self.feature_columns]
            
            # Generate the inference calculation
            prediction = self.model.predict(X_api)[0]
            
            # Formulate the response JSON format profile
            response = {
                "status": "SUCCESS",
                "prediction_type": "Continuous Traffic Volume Count",
                "predicted_vehicles": float(prediction)
            }
            return json.dumps(response, indent=2)
            
        except Exception as e:
            logger.error("API error processed during payload interpretation.", exc_info=True)
            return json.dumps({"status": "ERROR", "message": str(e)})

# Initialize the mock API server using our trained Random Forest classification/regression variables
api_server = TrafficModelAPIMock(trained_model=ensemble_reg, feature_columns=X_reg.columns)

# Test run the API simulation using a mock data row payload
sample_payload = json.dumps({
    "holiday": 0, "is_weekend": 0, "hour_sin": 0.5, "hour_cos": -0.86,
    "temp_scaled": 0.45, "clouds_scaled": 0.1, "is_freezing": 0, "is_raining": 0
})
api_response = api_server.predict(sample_payload)
print("\n[MLOPS MOCK API RESPONSE]:")
print(api_response)

# ==========================================
# 6.4 & 6.5: DRIFT MONITORING & ALERTING ENGINE
# ==========================================
def run_monitoring_drift_audit(incoming_traffic_stream, baseline_mean_volume):
    """
    Simulates production system health monitoring to alert engineering teams
    regarding sudden feature distribution data drift waves.
    """
    print("\n==================================================")
    print("PRODUCTION SYSTEMS HEALTH STATUS MONITORING PANEL")
    print("==================================================")
    
    current_stream_mean = np.mean(incoming_traffic_stream)
    drift_variance = np.abs(current_stream_mean - baseline_mean_volume) / baseline_mean_volume
    
    # Define a strict operational tolerance boundary of 15% drift
    DRIFT_THRESHOLD = 0.15
    
    print(f" -> Baseline Corridor Mean: {baseline_mean_volume:.1f} vehicles")
    print(f" -> Live Streamed Run Mean : {current_stream_mean:.1f} vehicles")
    print(f" -> Computed Signal Drift : {drift_variance * 100:.2f}%")
    
    if drift_variance > DRIFT_THRESHOLD:
        logger.warning(f"ALERT: System feature distribution drift detected! Variance level: {drift_variance*100:.2f}%")
        print("🚨 SYSTEM MONITORING STATUS: [ ALERT / REQUIRES INVESTIGATION ]")
        print(" -> Action required: Re-train active model arrays immediately due to highway environment shift.")
    else:
        logger.info("System monitoring dashboard check: PASS.")
        print("✅ SYSTEM MONITORING STATUS: [ PASS / NORMAL OPERATION ]")

# Run Evaluation Simulation A: Normal steady traffic flow
normal_traffic_stream = [3200, 3400, 3100, 3350, 2900]
run_monitoring_drift_audit(normal_traffic_stream, baseline_mean_volume=df_ml['traffic_volume'].mean())

# Run Evaluation Simulation B: Severe unexpected drift event (e.g., massive construction closure blockages)
drifted_traffic_stream = [600, 450, 500, 700, 400]
run_monitoring_drift_audit(drifted_traffic_stream, baseline_mean_volume=df_ml['traffic_volume'].mean())


ALERT: System feature distribution drift detected! Variance level: 83.74%


--- TASK 6: MLOPS DEPLOYMENT AND MONITORING SIMULATION ---

[MLOPS MOCK API RESPONSE]:
{
  "status": "SUCCESS",
  "prediction_type": "Continuous Traffic Volume Count",
  "predicted_vehicles": 4646.5
}

PRODUCTION SYSTEMS HEALTH STATUS MONITORING PANEL
 -> Baseline Corridor Mean: 3259.6 vehicles
 -> Live Streamed Run Mean : 3190.0 vehicles
 -> Computed Signal Drift : 2.14%
✅ SYSTEM MONITORING STATUS: [ PASS / NORMAL OPERATION ]

PRODUCTION SYSTEMS HEALTH STATUS MONITORING PANEL
 -> Baseline Corridor Mean: 3259.6 vehicles
 -> Live Streamed Run Mean : 530.0 vehicles
 -> Computed Signal Drift : 83.74%
🚨 SYSTEM MONITORING STATUS: [ ALERT / REQUIRES INVESTIGATION ]
 -> Action required: Re-train active model arrays immediately due to highway environment shift.
